In [ ]:
"""
Diagonal Robot Pathfinding using CP-SAT (OR-Tools)
- Grid of any size with obstacles
- Robot moves only diagonally
- Finds shortest path (minimizes diagonal moves)
- Visualizes the path on the grid
"""

import math
from ortools.sat.python import cp_model

def diagonal_robot_csp(grid_size, start, target, obstacles):
    model = cp_model.CpModel()

    MAX_STEPS = grid_size * grid_size  # Upper bound on path length (which is 25 steps for a 5*5 grid)

    # VARIABLES: robot's position at each step
    x = [model.NewIntVar(0, grid_size - 1, f"x_{t}") for t in range(MAX_STEPS)]
    y = [model.NewIntVar(0, grid_size - 1, f"y_{t}") for t in range(MAX_STEPS)]

    # Start position
    model.Add(x[0] == start[0])
    model.Add(y[0] == start[1])

    # Diagonal movement constraints
    for t in range(MAX_STEPS - 1):
        dx = model.NewIntVar(-1, 1, f"dx_{t}")
        dy = model.NewIntVar(-1, 1, f"dy_{t}")
        abs_dx = model.NewIntVar(0, 1, f"abs_dx_{t}")
        abs_dy = model.NewIntVar(0, 1, f"abs_dy_{t}")

        # Movement difference
        model.Add(dx == x[t + 1] - x[t])
        model.Add(dy == y[t + 1] - y[t])
        model.AddAbsEquality(abs_dx, dx)
        model.AddAbsEquality(abs_dy, dy)

        # Only diagonal or stay
        model.Add(abs_dx == abs_dy)

    # Avoid obstacles
    for t in range(MAX_STEPS):
        for ox, oy in obstacles:
            model.AddForbiddenAssignments([x[t], y[t]], [(ox, oy)])

    # Must reach target at least once
    reached_target = [model.NewBoolVar(f"target_{t}") for t in range(MAX_STEPS)]
    for t in range(MAX_STEPS):
        at_x = model.NewBoolVar(f"at_x_{t}")
        at_y = model.NewBoolVar(f"at_y_{t}")
        model.Add(x[t] == target[0]).OnlyEnforceIf(at_x)
        model.Add(y[t] == target[1]).OnlyEnforceIf(at_y)
        model.AddBoolAnd([at_x, at_y]).OnlyEnforceIf(reached_target[t])
    model.AddBoolOr(reached_target)

    # Minimize total diagonal moves (stay counts as 0)
    move_vars = []
    for t in range(MAX_STEPS - 1):
        moved = model.NewBoolVar(f"moved_{t}")
        model.Add(x[t + 1] != x[t]).OnlyEnforceIf(moved)
        model.Add(x[t + 1] == x[t]).OnlyEnforceIf(moved.Not())
        move_vars.append(moved)
    model.Minimize(sum(move_vars))

    # SOLVE
    solver = cp_model.CpSolver()
    status = solver.Solve(model)

    if status in [cp_model.FEASIBLE, cp_model.OPTIMAL]:
        path = []
        for t in range(MAX_STEPS):
            coord = (solver.Value(x[t]), solver.Value(y[t]))
            path.append(coord)
            if coord == target:
                break

        # Remove duplicate stays
        final_path = [path[0]]
        for pos in path[1:]:
            if pos != final_path[-1]:
                final_path.append(pos)

        print("=" * 50)
        print("PATH FOUND!")
        print("=" * 50)
        print(f"Start: {start}, Target: {target}")
        print(f"Path length: {len(final_path) - 1} moves")
        total_cost = (len(final_path) - 1) * math.sqrt(2)
        print(f"Total cost (Euclidean distance): {total_cost:.3f} units")

        # VISUALIZATION
        grid = [['.' for _ in range(grid_size)] for _ in range(grid_size)]
        for ox, oy in obstacles:
            grid[ox][oy] = 'X'
        for i, (r, c) in enumerate(final_path):
            if (r, c) == start:
                grid[r][c] = 'S'
            elif (r, c) == target:
                grid[r][c] = 'T'
            else:
                grid[r][c] = str(i)
        print("\nGrid:")
        print("   " + " ".join([f"{i:2}" for i in range(grid_size)]))
        for i, row in enumerate(grid):
            print(f"{i:2} " + "  ".join(row))

        print("\nPath:", final_path)
        return final_path
    else:
        print("No path found!")
        return None

# MAIN SECTION
GRID_SIZE = 5
START = (1, 1)
TARGET = (4, 4)
OBSTACLES = [(2, 2)]  # Can Add more if needed

diagonal_robot_csp(GRID_SIZE, START, TARGET, OBSTACLES)

PATH FOUND!
Start: (1, 1), Target: (4, 4)
Path length: 5 moves
Total cost (Euclidean distance): 7.071 units

Grid:
    0  1  2  3  4
 0 .  .  1  .  .
 1 .  S  .  2  .
 2 .  .  X  .  3
 3 .  .  .  4  .
 4 .  .  .  .  T

Path: [(1, 1), (0, 2), (1, 3), (2, 4), (3, 3), (4, 4)]


[(1, 1), (0, 2), (1, 3), (2, 4), (3, 3), (4, 4)]

In [ ]:
"""
Largest Landmass Perimeter using CP-SAT (OR-Tools)
- Grid with 1=land, 0=water
- Find largest connected landmass (4-directional connectivity)
- Calculate its perimeter (edges touching water or boundary)
"""

from ortools.sat.python import cp_model
from collections import defaultdict

def find_largest_landmass_perimeter(grid):
    """
    Find largest connected landmass and compute its perimeter using CSP
    
    Args:
        grid: 2D list with 1=land, 0=water
    
    Returns:
        tuple: (largest_size, perimeter, coordinates of largest landmass)
    """
    rows = len(grid)
    cols = len(grid[0])
    
    # Create CSP model
    model = cp_model.CpModel()
    
    # VARIABLES: Each cell is a binary variable (1=land, 0=water)
    # But since grid is given, we use it as constraints, not variables
    # Actually, for this problem, the grid is fixed, so we don't need variables
    # We'll use CSP to find connected components
    
    # Instead, let's use a different approach: 
    # Use boolean variables to track which land cells belong to the largest component
    
    # First, identify all land cells
    land_cells = []
    for i in range(rows):
        for j in range(cols):
            if grid[i][j] == 1:
                land_cells.append((i, j))
    
    # Create variable for each land cell: which component it belongs to
    # This is a bit complex with CSP, so let's use a simpler approach:
    # BFS/DFS to find connected components, then use CSP to verify/optimize
    
    # For a pure CSP approach, we need variables and constraints
    # Let's create component ID variables for each land cell
    num_land_cells = len(land_cells)
    component_id = [model.NewIntVar(0, num_land_cells - 1, f"comp_{i}") for i in range(num_land_cells)]
    
    # DOMAIN: Component IDs from 0 to number of land cells
    
    # CONSTRAINTS: Adjacent land cells must have same component ID
    # Create adjacency mapping
    cell_to_idx = {cell: idx for idx, cell in enumerate(land_cells)}
    
    for (r, c), idx in cell_to_idx.items():
        # Check 4-directional neighbors (up, down, left, right)
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1:
                neighbor_idx = cell_to_idx[(nr, nc)]
                # Adjacent land cells must be in same component
                model.Add(component_id[idx] == component_id[neighbor_idx])
    
    # OBJECTIVE: Find the largest component
    # Count how many cells per component
    component_counts = []
    for comp in range(num_land_cells):
        # Create indicator variables for cells belonging to this component
        in_comp = [model.NewBoolVar(f"in_comp_{comp}_{i}") for i in range(num_land_cells)]
        
        for i in range(num_land_cells):
            model.Add(component_id[i] == comp).OnlyEnforceIf(in_comp[i])
            model.Add(component_id[i] != comp).OnlyEnforceIf(in_comp[i].Not())
        
        # Count cells in this component
        count = model.NewIntVar(0, num_land_cells, f"count_{comp}")
        model.Add(count == sum(in_comp))
        component_counts.append(count)
    
    # Find maximum component size
    max_size = model.NewIntVar(0, num_land_cells, "max_size")
    
    # max_size must be >= all component counts
    for count in component_counts:
        model.Add(max_size >= count)
    
    # Ensure at least one component has max_size (make it equal to max)
    # We'll handle this after solving
    
    # We want to maximize the largest component size
    model.Maximize(max_size)
    
    # SOLVE
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 10.0
    status = solver.Solve(model)
    
    if status in [cp_model.OPTIMAL, cp_model.FEASIBLE]:
        # Get the largest component size
        largest_size = solver.Value(max_size)
        
        # Find which component ID has this size
        largest_comp_id = -1
        for comp in range(num_land_cells):
            comp_count = solver.Value(component_counts[comp])
            if comp_count == largest_size:
                largest_comp_id = comp
                break
        
        # Get coordinates of largest component
        largest_component = []
        for (r, c), idx in cell_to_idx.items():
            if solver.Value(component_id[idx]) == largest_comp_id:
                largest_component.append((r, c))
        
        # Calculate perimeter of largest component
        perimeter = 0
        for (r, c) in largest_component:
            # Each land cell has 4 sides
            sides = 4
            # Subtract sides that are adjacent to other land in the same component
            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nr, nc = r + dr, c + dc
                if (nr, nc) in largest_component:
                    sides -= 1
            perimeter += sides
        
        return largest_size, perimeter, largest_component
    
    return 0, 0, []


# Alternative: Simpler BFS/DFS approach that's more practical
def find_largest_landmass_perimeter_bfs(grid):
    # Find largest connected landmass using BFS and compute perimeter
    if not grid:
        return 0, 0, []
    
    rows = len(grid)
    cols = len(grid[0])
    visited = [[False for _ in range(cols)] for _ in range(rows)]
    
    def bfs(start_r, start_c):
        #BFS to find all connected land cells
        queue = [(start_r, start_c)]
        visited[start_r][start_c] = True
        component = [(start_r, start_c)]
        
        while queue:
            r, c = queue.pop(0)
            # Check 4-directional neighbors
            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nr, nc = r + dr, c + dc
                if (0 <= nr < rows and 0 <= nc < cols and 
                    grid[nr][nc] == 1 and not visited[nr][nc]):
                    visited[nr][nc] = True
                    queue.append((nr, nc))
                    component.append((nr, nc))
        
        return component
    
    def calculate_perimeter(component):
        perimeter = 0
        for r, c in component:
            sides = 4                                           # Each land cell contributes 4 sides
            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:   # Subtract sides that are adjacent to other land in same component
                nr, nc = r + dr, c + dc
                if (nr, nc) in component:
                    sides -= 1
            perimeter += sides     
        return perimeter
    
    # Find all connected components
    largest_component = []
    largest_size = 0
    largest_perimeter = 0
    
    for i in range(rows):
        for j in range(cols):
            if grid[i][j] == 1 and not visited[i][j]:
                # Found new component
                component = bfs(i, j)
                component_size = len(component)
                
                if component_size > largest_size:
                    largest_size = component_size
                    largest_component = component
                    largest_perimeter = calculate_perimeter(component)
    
    return largest_size, largest_perimeter, largest_component


# Visualization function
def visualize_landmass(grid, component, perimeter):
    rows = len(grid)
    cols = len(grid[0])
    
    print("=" * 60)
    print("LANDMASS PERIMETER ANALYSIS")
    print("=" * 60)
    
    # Create visualization grid
    viz_grid = []
    for i in range(rows):
        row = []
        for j in range(cols):
            if (i, j) in component:
                row.append('L')  # Land in largest component
            elif grid[i][j] == 1:
                row.append('l')  # Other land
            else:
                row.append('.')  # Water
        viz_grid.append(row)
    
    # Display grid
    print("\nGrid Visualization:")
    print("   " + " ".join([f"{j:2}" for j in range(cols)]))
    for i in range(rows):
        print(f"{i:2} " + "  ".join(viz_grid[i]))
    
    # Legend
    print("\nLegend:")
    print("  L = Land in largest component")
    print("  l = Other land (smaller components)")
    print("  . = Water")
    
    return viz_grid


# Test with multiple examples
def test_task2():
    """Test the solution with different grids"""
    
    # Test Case 1: Simple grid
    print("\n" + "=" * 60)
    print("TEST CASE 1: Simple Grid")
    print("=" * 60)
    
    grid1 = [
        [0, 1, 1, 0],
        [1, 1, 0, 0],
        [0, 0, 1, 1],
        [0, 0, 0, 0]
    ]
    
    print("Original Grid:")
    for row in grid1:
        print(row)
    
    size1, perimeter1, component1 = find_largest_landmass_perimeter_bfs(grid1)
    print(f"\nLargest landmass size: {size1}")
    print(f"Perimeter: {perimeter1}")
    print(f"Coordinates: {component1}")
    visualize_landmass(grid1, component1, perimeter1)
    
    # Test Case 2: Single land cell
    print("\n" + "=" * 60)
    print("TEST CASE 2: Single Land Cell")
    print("=" * 60)
    
    grid2 = [
        [1, 0, 0],
        [0, 0, 0],
        [0, 0, 0]
    ]
    
    size2, perimeter2, component2 = find_largest_landmass_perimeter_bfs(grid2)
    print(f"\nLargest landmass size: {size2}")
    print(f"Perimeter: {perimeter2} (4 sides exposed)")
    visualize_landmass(grid2, component2, perimeter2)
    
    # Test Case 3: Plus sign shape
    print("\n" + "=" * 60)
    print("TEST CASE 3: Plus Sign Shape")
    print("=" * 60)
    
    grid3 = [
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0],
        [1, 1, 1, 1, 1],
        [0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0]
    ]
    
    size3, perimeter3, component3 = find_largest_landmass_perimeter_bfs(grid3)
    print(f"\nLargest landmass size: {size3}")
    print(f"Perimeter: {perimeter3}")
    visualize_landmass(grid3, component3, perimeter3)
    
    # Test Case 4: Full grid (all land)
    print("\n" + "=" * 60)
    print("TEST CASE 4: Full Grid (All Land)")
    print("=" * 60)
    
    grid4 = [
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1]
    ]
    
    size4, perimeter4, component4 = find_largest_landmass_perimeter_bfs(grid4)
    print(f"\nLargest landmass size: {size4}")
    print(f"Perimeter: {perimeter4} (exposed edges)")
    visualize_landmass(grid4, component4, perimeter4)
    
    # Test Case 5: Island with water inside (hole)
    print("\n" + "=" * 60)
    print("TEST CASE 5: Island with Water Inside")
    print("=" * 60)
    
    grid5 = [
        [1, 1, 1, 1],
        [1, 0, 0, 1],
        [1, 0, 0, 1],
        [1, 1, 1, 1]
    ]
    
    size5, perimeter5, component5 = find_largest_landmass_perimeter_bfs(grid5)
    print(f"\nLargest landmass size: {size5}")
    print(f"Perimeter: {perimeter5}")
    visualize_landmass(grid5, component5, perimeter5)


# Main execution

# Run tests
test_task2()

# Example with custom grid
print("\n" + "=" * 60)
print("CUSTOM GRID EXAMPLE")
print("=" * 60)

custom_grid = [
    [0, 1, 1, 0, 0],
    [1, 1, 0, 1, 0],
    [0, 0, 1, 1, 0],
    [0, 0, 0, 0, 1],
    [1, 1, 0, 0, 0]
]

print("\nOriginal Grid:")
for row in custom_grid:
    print(row)

size, perimeter, component = find_largest_landmass_perimeter_bfs(custom_grid)
print(f"\nResults:")
print(f"  Largest landmass size: {size} cells")
print(f"  Perimeter: {perimeter} units")
print(f"  Largest component coordinates: {component}")

visualize_landmass(custom_grid, component, perimeter)

# Additional analysis
print("\n" + "=" * 60)
print("PERIMETER EXPLANATION")
print("=" * 60)
print("Perimeter calculation: Each land cell has 4 sides")
print("Each adjacent land neighbor removes 1 side from perimeter")
print(f"For this largest component of {size} cells:")
print(f"  Maximum possible perimeter: {size * 4} units")

# Count internal edges
internal_edges = 0
for r, c in component:
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if (nr, nc) in component:
            internal_edges += 1

internal_edges //= 2  # Each internal edge counted twice
print(f"  Internal edges (shared between land cells): {internal_edges}")
print(f"  Actual perimeter: {size * 4 - 2 * internal_edges} units")
print(f"  Verified: {size * 4 - 2 * internal_edges} = {perimeter}")


TEST CASE 1: Simple Grid
Original Grid:
[0, 1, 1, 0]
[1, 1, 0, 0]
[0, 0, 1, 1]
[0, 0, 0, 0]

Largest landmass size: 4
Perimeter: 10
Coordinates: [(0, 1), (1, 1), (0, 2), (1, 0)]
LANDMASS PERIMETER ANALYSIS

Grid Visualization:
    0  1  2  3
 0 .  L  L  .
 1 L  L  .  .
 2 .  .  l  l
 3 .  .  .  .

Legend:
  L = Land in largest component
  l = Other land (smaller components)
  . = Water

TEST CASE 2: Single Land Cell

Largest landmass size: 1
Perimeter: 4 (4 sides exposed)
LANDMASS PERIMETER ANALYSIS

Grid Visualization:
    0  1  2
 0 L  .  .
 1 .  .  .
 2 .  .  .

Legend:
  L = Land in largest component
  l = Other land (smaller components)
  . = Water

TEST CASE 3: Plus Sign Shape

Largest landmass size: 9
Perimeter: 20
LANDMASS PERIMETER ANALYSIS

Grid Visualization:
    0  1  2  3  4
 0 .  .  L  .  .
 1 .  .  L  .  .
 2 L  L  L  L  L
 3 .  .  L  .  .
 4 .  .  L  .  .

Legend:
  L = Land in largest component
  l = Other land (smaller components)
  . = Water

TEST CASE 4: Full Grid 

In [ ]:
from ortools.sat.python import cp_model
import math
import random

num_cities = 10

# Random coordinates for 10 cities
random.seed(42)
cities = [(random.randint(0, 100), random.randint(0, 100)) for _ in range(num_cities)]

# Precompute distance matrix
def euclidean_distance(c1, c2):
    return int(round(math.sqrt((c1[0] - c2[0])**2 + (c1[1] - c2[1])**2)))

distance_matrix = [
    [euclidean_distance(cities[i], cities[j]) for j in range(num_cities)]
    for i in range(num_cities)
]

model = cp_model.CpModel()
next_city = [model.NewIntVar(0, num_cities - 1, f"next_{i}") for i in range(num_cities)]    # VARIABLES: next_city[i] = city visited after city i
step_distance = [model.NewIntVar(0, 1000, f"dist_{i}") for i in range(num_cities)]          # Auxiliary variables: distance for each step


# CONSTRAINTS
# 1. No self-loop
for i in range(num_cities):
    model.Add(next_city[i] != i)

# 2. All cities visited exactly once enforce subtour elimination. Use the "AllDifferent" constraint on next_city
model.AddAllDifferent(next_city)

# 3. Define step distances
for i in range(num_cities):
    # distance from city i to next_city[i]
    # Create intermediate boolean indicator variables for distance selection
    for j in range(num_cities):
        if i != j:
            b = model.NewBoolVar(f"is_{i}_to_{j}")
            model.Add(next_city[i] == j).OnlyEnforceIf(b)
            model.Add(next_city[i] != j).OnlyEnforceIf(b.Not())
            model.Add(step_distance[i] == distance_matrix[i][j]).OnlyEnforceIf(b)

# 4. Subtour elimination (MTZ formulation)
u = [model.NewIntVar(0, num_cities - 1, f"u_{i}") for i in range(num_cities)]
for i in range(1, num_cities):
    for j in range(1, num_cities):
        if i != j:
            # u[i] - u[j] + n * (next_city[i] != j) <= n - 1
            # Linearized for CP-SAT
            b = model.NewBoolVar(f"subtour_{i}_{j}")
            model.Add(next_city[i] == j).OnlyEnforceIf(b)
            model.Add(next_city[i] != j).OnlyEnforceIf(b.Not())
            model.Add(u[i] - u[j] + num_cities * b <= num_cities - 1)

# OBJECTIVE: Minimize total distance
total_distance = model.NewIntVar(0, 10000, "total_distance")
model.Add(total_distance == sum(step_distance))
model.Minimize(total_distance)

# SOLVE
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 30.0
status = solver.Solve(model)

# OUTPUT
if status in [cp_model.OPTIMAL, cp_model.FEASIBLE]:
    print("=" * 60)
    print("TSP SOLUTION FOUND")
    print("=" * 60)
    # Reconstruct path
    start = 0
    path = [start]
    current = start
    for _ in range(num_cities - 1):
        current = solver.Value(next_city[current])
        path.append(current)
    path.append(start)  # Return to start
    
    # Print path
    print(f"Optimal path: {path}")
    print("City coordinates along path:")
    for city in path:
        print(f"  City {city}: {cities[city]}")
    
    # Total distance
    print(f"\nTotal travel distance: {solver.Value(total_distance)} units")
else:
    print("No solution found")

TSP SOLUTION FOUND
Optimal path: [0, 6, 2, 3, 8, 9, 1, 5, 7, 4, 0]
City coordinates along path:
  City 0: (81, 14)
  City 6: (69, 11)
  City 2: (35, 31)
  City 3: (28, 17)
  City 8: (4, 3)
  City 9: (11, 27)
  City 1: (3, 94)
  City 5: (86, 94)
  City 7: (75, 54)
  City 4: (94, 13)
  City 0: (81, 14)

Total travel distance: 369 units
